# 03 — Feature Engineering, Bunching Label & Database Storage

Joins cleaned location data with scheduled headway, computes the **observed headway**
between consecutive vehicles on each route (via a windowed `lag()`), derives the
**bunching label**, aggregates to route level, and persists everything to SQLite using
**parameterised queries only**.


In [1]:
import os, sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = r'C:\Hadoop'

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.appName("BusBunching_03_FeatureEngineering").master("local[4]").config("spark.ui.showConsoleProgress", "false").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

location_clean = spark.read.option("header", True).option("inferSchema", True).csv("data/cleaned_locations.csv")
location_clean = location_clean.withColumn("recorded_at_time", F.to_timestamp("recorded_at_time"))
location_clean = location_clean.withColumn("line_ref", F.col("line_ref").cast("string"))

routes_headway = spark.read.option("header", True).option("inferSchema", True).csv(r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\data\routes_scheduled_headway.csv")
routes_headway = routes_headway.withColumn("line_ref", F.col("line_ref").cast("string"))

print("Locations:", location_clean.count(), "| Routes:", routes_headway.count())


Locations: 762 | Routes: 106


Locations: 762 | Routes: 106


## 3.1 Compute observed headway per route (windowed lag — O(n log n) per route)

In [3]:
route_window = Window.partitionBy("line_ref").orderBy("recorded_at_time")

headway_df = location_clean.withColumn(
    "prev_time", F.lag("recorded_at_time").over(route_window)
).withColumn(
    "observed_headway_sec",
    F.unix_timestamp("recorded_at_time") - F.unix_timestamp("prev_time")
).dropna(subset=["observed_headway_sec"])

headway_df.select("line_ref", "vehicle_ref", "recorded_at_time", "observed_headway_sec").show(10)


+--------+-----------+-------------------+--------------------+
|line_ref|vehicle_ref|   recorded_at_time|observed_headway_sec|
+--------+-----------+-------------------+--------------------+
|       1| SCCU-15854|2026-07-28 14:00:34|                8791|
|       1| SCCU-15578|2026-07-28 18:23:46|               15792|
|       1| SCCU-36990|2026-07-28 22:42:54|               15548|
|       1| SCCU-37041|2026-07-28 22:47:55|                 301|
|       1| SCCU-12211|2026-07-28 23:09:58|                1323|
|       1| SCCU-37484|2026-07-28 23:15:21|                 323|
|       1| SCCU-15729|2026-07-28 23:19:39|                 258|
|       1| SCCU-15300|2026-07-28 23:20:49|                  70|
|       1| SCCU-36820|2026-07-28 23:33:19|                 750|
|       1| SCCU-36103|2026-07-28 23:47:43|                 864|
+--------+-----------+-------------------+--------------------+
only showing top 10 rows



## 3.2 Join with scheduled headway (broadcast join — small lookup table)

In [4]:
headway_joined = headway_df.join(
    F.broadcast(routes_headway.select("line_ref", "scheduled_headway_sec")),
    on="line_ref", how="left"
).withColumn(
    "headway_ratio", F.col("observed_headway_sec") / F.col("scheduled_headway_sec")
).withColumn(
    "is_bunching", (F.col("headway_ratio") < 0.3).cast("int")
)

headway_joined.select("line_ref", "observed_headway_sec", "scheduled_headway_sec",
                       "headway_ratio", "is_bunching").show(10)

print("Overall bunching rate:",
      headway_joined.selectExpr("avg(is_bunching) as rate").collect()[0]["rate"])


+--------+--------------------+---------------------+------------------+-----------+
|line_ref|observed_headway_sec|scheduled_headway_sec|     headway_ratio|is_bunching|
+--------+--------------------+---------------------+------------------+-----------+
|       1|                8791|    339.3559300925734|25.904954711125544|          0|
|       1|               15792|    339.3559300925734| 46.53521155705774|          0|
|       1|               15548|    339.3559300925734| 45.81620246258446|          0|
|       1|                 301|    339.3559300925734|0.8869743337559765|          0|
|       1|                1323|    339.3559300925734|3.8985616065088267|          0|
|       1|                 323|    339.3559300925734|0.9518030226019283|          0|
|       1|                 258|    339.3559300925734|0.7602637146479798|          0|
|       1|                  70|    339.3559300925734|0.2062731008734829|          1|
|       1|                 750|    339.3559300925734| 2.210068937

Overall bunching rate: 0.2613333333333333


## 3.2b Engineer genuine *leading* predictors (not the same-instant headway)


In [5]:
route_window2 = Window.partitionBy("line_ref").orderBy("recorded_at_time")

headway_joined = headway_joined.withColumn(
    "prev_headway_sec", F.lag("observed_headway_sec").over(route_window2)
).withColumn(
    "hour_of_day", F.hour("recorded_at_time")
).dropna(subset=["prev_headway_sec"])

headway_joined.select("line_ref", "hour_of_day", "prev_headway_sec",
                       "observed_headway_sec", "is_bunching").show(10)


+--------+-----------+----------------+--------------------+-----------+
|line_ref|hour_of_day|prev_headway_sec|observed_headway_sec|is_bunching|
+--------+-----------+----------------+--------------------+-----------+
|       1|         18|            8791|               15792|          0|
|       1|         22|           15792|               15548|          0|
|       1|         22|           15548|                 301|          0|
|       1|         23|             301|                1323|          0|
|       1|         23|            1323|                 323|          0|
|       1|         23|             323|                 258|          0|
|       1|         23|             258|                  70|          1|
|       1|         23|              70|                 750|          0|
|       1|         23|             750|                 864|          0|
|       1|          0|             864|                 850|          0|
+--------+-----------+----------------+------------

## 3.3 Aggregate to route level (used by the ML notebooks)

In [6]:
route_level = headway_joined.groupBy("line_ref").agg(
    F.count("*").alias("n_observations"),
    F.avg("observed_headway_sec").alias("avg_observed_headway_sec"),
    F.avg("headway_ratio").alias("avg_headway_ratio"),
    F.stddev("observed_headway_sec").alias("headway_std_sec"),
    F.avg("is_bunching").alias("bunching_rate")
)
route_level.show()

route_level_pdf = route_level.toPandas()
route_level_pdf.to_csv("data/route_level_features.csv", index=False)

event_level_pdf = headway_joined.select(
    "line_ref", "vehicle_ref", "recorded_at_time",
    "hour_of_day", "prev_headway_sec",                          # genuine leading predictors
    "observed_headway_sec", "scheduled_headway_sec", "headway_ratio",  # kept for reference/audit only
    "is_bunching"
).toPandas()
event_level_pdf.to_csv("data/event_level_features.csv", index=False)

print("Saved data/route_level_features.csv and data/event_level_features.csv")


+--------+--------------+------------------------+------------------+------------------+-------------------+
|line_ref|n_observations|avg_observed_headway_sec| avg_headway_ratio|   headway_std_sec|      bunching_rate|
+--------+--------------+------------------------+------------------+------------------+-------------------+
|       1|            27|       2053.222222222222|  6.05035020800173| 4021.127791795321|0.07407407407407407|
|     100|             4|                  5842.0| 7.616352921485893| 10049.49872713394|                0.0|
|     104|             6|      3177.6666666666665|1.9099281108617436|1541.9554684447496|0.16666666666666666|
|     109|             1|                   114.0| 0.068686272857504|              NULL|                1.0|
|     10A|            65|       535.3076923076923|              NULL|2051.5992765026394|               NULL|
|     111|            12|                  2492.5| 6.112569586747408|3144.3377247478884|                0.0|
|     125|         

Saved data/route_level_features.csv and data/event_level_features.csv


## 3.4 Persist to SQLite via parameterised queries (no string concatenation)

In [8]:
import sqlite3

DB_PATH = "bus_bunching.db"
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS routes (
    line_ref TEXT PRIMARY KEY,
    route_name TEXT,
    scheduled_headway_sec REAL
)''')

cur.execute('''
CREATE TABLE IF NOT EXISTS bunching_events (
    line_ref TEXT,
    vehicle_ref TEXT,
    recorded_at_time TEXT,
    observed_headway_sec REAL,
    scheduled_headway_sec REAL,
    headway_ratio REAL,
    is_bunching INTEGER
)''')
conn.commit()

# Parameterised inserts — prevents SQL injection
routes_records = pd.read_csv(r"C:\Users\ACER\OneDrive\Documents\Big Data Programming Project Coursework\data\routes_scheduled_headway.csv")[
    ["line_ref", "route_name", "scheduled_headway_sec"]
].values.tolist()
cur.executemany("INSERT OR REPLACE INTO routes VALUES (?, ?, ?)", routes_records)

event_records = event_level_pdf[[
    "line_ref", "vehicle_ref", "recorded_at_time", "observed_headway_sec",
    "scheduled_headway_sec", "headway_ratio", "is_bunching"
]].astype(str).values.tolist()
cur.executemany(
    "INSERT INTO bunching_events VALUES (?, ?, ?, ?, ?, ?, ?)", event_records
)
conn.commit()
conn.close()
print(f"Inserted {len(routes_records)} routes and {len(event_records)} bunching events into {DB_PATH}")


Inserted 106 routes and 573 bunching events into bus_bunching.db
